# Silver Processing — Customers

Cleans `ecommerce.bronze.bronze_customers` (300,000 rows) into `ecommerce.silver.silver_customers`:

* **All business columns**: trim spaces and strip trailing special characters
* **`phone`**: converted from double (scientific notation, `.0` artifact) to a clean string of digits — e.g. `9.18187043562E11` → `"918187043562"`; nulls (30,000) preserved
* **Invalid rows dropped**: 300 rows (0.1%) have a NULL `customer_id` — unusable as dimension keys, removed (still recoverable from bronze)
* All remaining `customer_id` values are unique (verified: 299,700 distinct)
* **Metadata columns** (`source_file`, `ingested_at`) pass through untouched

In [0]:
from pyspark.sql.functions import col, trim, regexp_replace

bronze_table = "ecommerce.bronze.bronze_customers"
target_table = "ecommerce.silver.silver_customers"

df_bronze_customers = spark.table(bronze_table)

# Clean a business column: trim spaces and strip any trailing special characters
def clean(column_name):
    return regexp_replace(trim(col(column_name)), "[^A-Za-z0-9]+$", "")

df_silver_customers = (
    df_bronze_customers
    # Drop rows with no customer_id — invalid dimension records (300 of 300,000)
    .where(col("customer_id").isNotNull())
    .withColumn("customer_id", clean("customer_id"))
    .withColumn("country_code", clean("country_code"))
    .withColumn("country", clean("country"))
    .withColumn("state", clean("state"))
    # phone: double -> bigint -> string of digits (removes scientific notation / .0)
    .withColumn("phone", col("phone").cast("bigint").cast("string"))
)

# Idempotent full load into the silver table
(
    df_silver_customers.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
display(spark.table("ecommerce.silver.silver_customers").limit(10))

In [0]:
# Check 1: row count and key uniqueness — expect 299,700 rows, all customer_id unique
spark.sql("""
    SELECT
      COUNT(*) AS rows,
      COUNT(DISTINCT customer_id) AS distinct_ids,
      COUNT_IF(customer_id IS NULL) AS null_ids,
      COUNT_IF(phone IS NULL) AS null_phones,
      COUNT_IF(phone RLIKE '[^0-9]') AS non_digit_phones
    FROM ecommerce.silver.silver_customers
""").show()

# Check 2: no residual padding or trailing special characters in business columns
spark.sql("""
    SELECT COUNT(*) AS dirty_rows
    FROM ecommerce.silver.silver_customers
    WHERE customer_id != trim(customer_id)
       OR customer_id RLIKE '[^A-Za-z0-9]$'
       OR country != trim(country)
       OR state != trim(state)
       OR country_code != trim(country_code)
""").show()